# LESSON 6.1: A Model of the Image Degradation/Restoration Process
## Image Restoration

In this lesson:
- What is image restoration and how it differs from image enhancement
- The degradation model: $g(x,y) = h(x,y) \star f(x,y) + \eta(x,y)$
- Understanding degradation in both spatial and frequency domains
- Types of degradation: blur, noise, and their combinations
- Overview of restoration approaches
- Simulating degraded images for restoration experiments

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Image Restoration vs. Image Enhancement

**Image Enhancement** (Chapter 3-4):
- Subjective process: make the image "look better" to a human observer
- No mathematical model of degradation needed
- Examples: contrast stretching, histogram equalization, sharpening

**Image Restoration** (Chapter 5):
- Objective process: recover the original image from a degraded observation
- Requires a **mathematical model** of the degradation process
- Uses knowledge of the degradation to **invert** the process
- Examples: deblurring, denoising, motion correction

### Key Difference:
- Enhancement: "I want the image to look better" (subjective)
- Restoration: "I know what happened to the image, and I want to undo it" (objective)

### Clinical Relevance:
In biomedical imaging, restoration is critical because:
- Patient motion during MRI causes blur
- Low-dose CT produces noisy images
- Optical aberrations in microscopy degrade cell images
- Accurate diagnosis depends on image fidelity, not just appearance

---
## 2. The Degradation Model

The general degradation model describes how an ideal image $f(x,y)$ is transformed into a degraded observation $g(x,y)$:

### Spatial Domain:
$$\boxed{g(x,y) = h(x,y) \star f(x,y) + \eta(x,y)}$$

### Frequency Domain:
$$\boxed{G(u,v) = H(u,v) \cdot F(u,v) + N(u,v)}$$

Where:
- $f(x,y)$ / $F(u,v)$ = **original (undegraded) image**
- $h(x,y)$ / $H(u,v)$ = **degradation function** (Point Spread Function / Optical Transfer Function)
- $\eta(x,y)$ / $N(u,v)$ = **additive noise**
- $g(x,y)$ / $G(u,v)$ = **observed (degraded) image**
- $\star$ denotes **convolution**

### Block Diagram:
```
f(x,y) ---> [ H (Degradation) ] ---(+)---> g(x,y)
                                     ^
                                     |
                                  η(x,y)
                                  (Noise)
```

### Goal of Restoration:
Given the degraded image $g(x,y)$ and knowledge of $H$ and/or $\eta$, estimate $\hat{f}(x,y)$ that is as close as possible to the original $f(x,y)$.

In [ ]:
# Helper function: create a test image (biomedical phantom)
def create_test_image(size=256):
    """
    Create a synthetic biomedical phantom image with organs, vessels, and structures.
    
    Returns:
        2D numpy array of shape (size, size), values in [0, 255]
    """
    img = np.ones((size, size), dtype=np.float64) * 30  # dark background
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    # Body contour
    body = ((X - cx) / 100) ** 2 + ((Y - cy) / 80) ** 2 <= 1
    img[body] = 120
    
    # Organ 1 (bright)
    organ1 = ((X - cx + 30) / 35) ** 2 + ((Y - cy + 10) / 45) ** 2 <= 1
    img[organ1] = 170
    
    # Organ 2 (dark)
    organ2 = ((X - cx - 35) / 25) ** 2 + ((Y - cy - 15) / 30) ** 2 <= 1
    img[organ2] = 80
    
    # Small bright spots (calcifications)
    for (sx, sy, sr) in [(cx-20, cy+30, 5), (cx+40, cy-25, 4), (cx-50, cy-20, 3)]:
        spot = (X - sx) ** 2 + (Y - sy) ** 2 <= sr ** 2
        img[spot] = 240
    
    # Fine structures (vessels)
    for i in range(cy-40, cy+40):
        j = int(cx + 20 * np.sin(2 * np.pi * i / 50))
        if 0 <= j < size and 0 <= i < size:
            img[i, max(0,j-1):min(size,j+2)] = 200
    
    return img


# Create the original image
original = create_test_image(256)

plt.figure(figsize=(6, 6))
plt.imshow(original, cmap='gray', vmin=0, vmax=255)
plt.title('Original Image f(x,y) - Biomedical Phantom', fontsize=13)
plt.colorbar()
plt.show()

---
## 3. Components of the Degradation Model

### 3.1 The Degradation Function H (Point Spread Function)

The **Point Spread Function (PSF)** $h(x,y)$ describes how the imaging system blurs a single point of light:

| Degradation Type | PSF Description | Common Cause |
|---|---|---|
| **No degradation** | Delta function $\delta(x,y)$ | Perfect system |
| **Gaussian blur** | $h = \frac{1}{2\pi\sigma^2} e^{-(x^2+y^2)/(2\sigma^2)}$ | Out-of-focus, atmospheric |
| **Motion blur** | Line along motion direction | Camera/patient motion |
| **Uniform blur** | Constant within a disk | Defocus aberration |

### 3.2 Additive Noise η(x,y)

Noise is **random** and cannot be predicted from the image content:

| Noise Type | Description | Source |
|---|---|---|
| **Gaussian** | Bell-shaped distribution | Sensor electronics |
| **Salt & Pepper** | Random black/white pixels | Transmission errors |
| **Poisson** | Signal-dependent | Low photon count (X-ray, nuclear medicine) |
| **Speckle** | Multiplicative | Ultrasound, SAR |

### 3.3 Assumption: Linear, Position-Invariant (LPI) System

The degradation model $g = h \star f + \eta$ assumes:
- **Linearity**: $H\{a f_1 + b f_2\} = a H\{f_1\} + b H\{f_2\}$
- **Position-Invariance (Shift-Invariance)**: The PSF is the same everywhere in the image

These assumptions enable us to work in the frequency domain using the convolution theorem.

In [ ]:
# Demonstrate different types of degradation functions (PSFs)

def create_gaussian_psf(size, sigma):
    """Create a Gaussian Point Spread Function."""
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    psf = np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sigma**2))
    psf /= psf.sum()  # normalize
    return psf


def create_motion_psf(size, length, angle_deg=0):
    """Create a motion blur PSF along a given angle."""
    psf = np.zeros((size, size), dtype=np.float64)
    cx, cy = size // 2, size // 2
    angle_rad = np.deg2rad(angle_deg)
    for t in np.linspace(-length/2, length/2, max(length*2, 100)):
        x = int(round(cx + t * np.cos(angle_rad)))
        y = int(round(cy + t * np.sin(angle_rad)))
        if 0 <= x < size and 0 <= y < size:
            psf[y, x] = 1.0
    psf /= psf.sum()  # normalize
    return psf


def create_disk_psf(size, radius):
    """Create a uniform disk (defocus) PSF."""
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    psf = np.zeros((size, size), dtype=np.float64)
    mask = (X - cx)**2 + (Y - cy)**2 <= radius**2
    psf[mask] = 1.0
    psf /= psf.sum()
    return psf


# Create PSFs
psf_size = 64
psf_delta = np.zeros((psf_size, psf_size))
psf_delta[psf_size//2, psf_size//2] = 1.0

psf_gauss = create_gaussian_psf(psf_size, sigma=5)
psf_motion = create_motion_psf(psf_size, length=20, angle_deg=30)
psf_disk = create_disk_psf(psf_size, radius=8)

# Visualize PSFs
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

psfs = [psf_delta, psf_gauss, psf_motion, psf_disk]
titles = ['Delta (No Blur)', 'Gaussian (σ=5)', 'Motion (L=20, θ=30°)', 'Disk (r=8)']

for i, (psf, title) in enumerate(zip(psfs, titles)):
    # PSF in spatial domain
    axes[0, i].imshow(psf, cmap='hot')
    axes[0, i].set_title(f'PSF: {title}', fontsize=11)
    axes[0, i].axis('off')
    
    # OTF (magnitude) in frequency domain
    H = np.fft.fftshift(np.fft.fft2(psf))
    axes[1, i].imshow(np.log1p(np.abs(H)), cmap='hot')
    axes[1, i].set_title(f'OTF: |H(u,v)|', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Spatial Domain\nh(x,y)', fontsize=12)
axes[1, 0].set_ylabel('Frequency Domain\n|H(u,v)|', fontsize=12)

plt.suptitle('Point Spread Functions (PSFs) and Their Optical Transfer Functions (OTFs)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Delta PSF: No degradation (perfect imaging system)")
print("Gaussian PSF: Out-of-focus blur, atmospheric turbulence")
print("Motion PSF: Camera or patient movement during exposure")
print("Disk PSF: Uniform defocus (circular aperture)")

In [ ]:
# Apply degradation function to the original image

def apply_degradation(image, psf):
    """
    Apply a degradation function (PSF) to an image via frequency domain convolution.
    
    Parameters:
        image: 2D numpy array (original image)
        psf: 2D numpy array (point spread function, same size as image)
    Returns:
        Degraded image (blurred, no noise)
    """
    F = np.fft.fft2(image)
    H = np.fft.fft2(psf)
    G = F * H
    g = np.real(np.fft.ifft2(G))
    return g


def pad_psf_to_image_size(psf, image_shape):
    """
    Zero-pad a small PSF to the same size as the image.
    The PSF center is placed at (0,0) for correct DFT convolution.
    """
    padded = np.zeros(image_shape, dtype=np.float64)
    ph, pw = psf.shape
    # Place PSF center at (0,0) using circular shift
    for i in range(ph):
        for j in range(pw):
            ni = (i - ph // 2) % image_shape[0]
            nj = (j - pw // 2) % image_shape[1]
            padded[ni, nj] = psf[i, j]
    return padded


# Create PSFs at image size
M, N = original.shape
psf_g = pad_psf_to_image_size(create_gaussian_psf(64, sigma=3), (M, N))
psf_m = pad_psf_to_image_size(create_motion_psf(64, length=25, angle_deg=0), (M, N))
psf_d = pad_psf_to_image_size(create_disk_psf(64, radius=6), (M, N))

# Apply blur only (no noise)
blurred_gauss = apply_degradation(original, psf_g)
blurred_motion = apply_degradation(original, psf_m)
blurred_disk = apply_degradation(original, psf_d)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original f(x,y)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(blurred_gauss, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Gaussian Blur (σ=3)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(blurred_motion, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Motion Blur (L=25, θ=0°)', fontsize=12)
axes[2].axis('off')

axes[3].imshow(blurred_disk, cmap='gray', vmin=0, vmax=255)
axes[3].set_title('Disk Blur (r=6)', fontsize=12)
axes[3].axis('off')

plt.suptitle('Effect of Different Degradation Functions (Blur Only, No Noise)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. The Complete Degradation: Blur + Noise

In practice, degraded images suffer from **both** blur and noise simultaneously:

$$g(x,y) = h(x,y) \star f(x,y) + \eta(x,y)$$

The combination makes restoration much harder than dealing with either problem alone:
- Deblurring can **amplify noise**
- Denoising can **further blur** the image
- The restoration filter must **balance** both objectives

In [ ]:
# Simulate the full degradation model: g = h*f + noise

def add_gaussian_noise(image, mean=0, sigma=10):
    """Add Gaussian noise to an image."""
    np.random.seed(42)
    noise = np.random.normal(mean, sigma, image.shape)
    noisy = image + noise
    return np.clip(noisy, 0, 255), noise


# Scenario 1: Noise only (no blur)
noisy_only, noise = add_gaussian_noise(original, sigma=15)

# Scenario 2: Blur only (no noise)
blur_only = blurred_gauss.copy()

# Scenario 3: Blur + Noise (full degradation)
blur_and_noise, _ = add_gaussian_noise(blurred_gauss, sigma=15)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Top row: images
axes[0, 0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original f(x,y)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_only, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Noise Only (σ=15)', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(blur_only, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title('Blur Only (Gaussian σ=3)', fontsize=12)
axes[0, 2].axis('off')

axes[0, 3].imshow(blur_and_noise, cmap='gray', vmin=0, vmax=255)
axes[0, 3].set_title('Blur + Noise\n(Full Degradation)', fontsize=12)
axes[0, 3].axis('off')

# Bottom row: intensity profiles
row_idx = 128
for ax in axes[1, :]:
    ax.set_xlabel('Column', fontsize=10)
    ax.set_ylabel('Intensity', fontsize=10)
    ax.grid(True, alpha=0.3)

axes[1, 0].plot(original[row_idx, :], 'k-', linewidth=1.5)
axes[1, 0].set_title(f'Original Row {row_idx}', fontsize=11)

axes[1, 1].plot(noisy_only[row_idx, :], 'r-', linewidth=1)
axes[1, 1].set_title('Noise Only', fontsize=11)

axes[1, 2].plot(blur_only[row_idx, :], 'b-', linewidth=1.5)
axes[1, 2].set_title('Blur Only', fontsize=11)

axes[1, 3].plot(blur_and_noise[row_idx, :], 'm-', linewidth=1)
axes[1, 3].set_title('Blur + Noise', fontsize=11)

plt.suptitle('Components of Image Degradation: Noise, Blur, and Their Combination',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Noise only: Sharp edges preserved but random fluctuations added")
print("Blur only: Smooth result but edges and details are lost")
print("Blur + Noise: The worst case - both detail loss AND random corruption")

In [ ]:
# Visualize the degradation in the frequency domain

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

images = [original, noisy_only, blur_only, blur_and_noise]
titles_spatial = ['Original', 'Noise Only', 'Blur Only', 'Blur + Noise']
titles_freq = ['|F(u,v)|', '|F(u,v) + N(u,v)|', '|H·F(u,v)|', '|H·F(u,v) + N(u,v)|']

for i, (img, t_s, t_f) in enumerate(zip(images, titles_spatial, titles_freq)):
    axes[0, i].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(t_s, fontsize=12)
    axes[0, i].axis('off')
    
    F = np.fft.fftshift(np.fft.fft2(img))
    spectrum = np.log1p(np.abs(F))
    axes[1, i].imshow(spectrum, cmap='hot')
    axes[1, i].set_title(t_f, fontsize=12)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Spatial Domain', fontsize=12)
axes[1, 0].set_ylabel('Frequency Domain', fontsize=12)

plt.suptitle('Degradation Effects in Spatial and Frequency Domains',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Noise: Adds energy across ALL frequencies (flat spectrum addition)")
print("Blur: Attenuates HIGH frequencies (spectrum concentrates at center)")
print("Blur+Noise: High frequencies dominated by noise, low by signal")

---
## 5. Signal-to-Noise Ratio (SNR)

A fundamental measure of image quality is the **Signal-to-Noise Ratio**:

$$\text{SNR} = \frac{\text{Signal Power}}{\text{Noise Power}} = \frac{\sum |f(x,y)|^2}{\sum |\eta(x,y)|^2}$$

Often expressed in **decibels (dB)**:

$$\text{SNR}_{\text{dB}} = 10 \log_{10}\left(\frac{\sum |f(x,y)|^2}{\sum |\eta(x,y)|^2}\right)$$

### Another common metric: Peak Signal-to-Noise Ratio (PSNR)

$$\text{PSNR} = 10 \log_{10}\left(\frac{\text{MAX}^2}{\text{MSE}}\right) \quad \text{dB}$$

Where:
- MAX = maximum possible pixel value (e.g., 255)
- MSE = Mean Squared Error between original and degraded images

### Typical PSNR values:
- PSNR > 40 dB: Excellent quality, almost indistinguishable from original
- PSNR 30-40 dB: Good quality, minor artifacts
- PSNR 20-30 dB: Poor quality, noticeable degradation
- PSNR < 20 dB: Very poor quality

In [ ]:
# Compute and visualize quality metrics for different noise levels

def compute_psnr(original, degraded):
    """Compute Peak Signal-to-Noise Ratio in dB."""
    mse = np.mean((original - degraded) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255.0 ** 2 / mse)


def compute_mse(original, degraded):
    """Compute Mean Squared Error."""
    return np.mean((original - degraded) ** 2)


# Different noise levels
sigma_values = [5, 10, 20, 40, 80]

fig, axes = plt.subplots(2, len(sigma_values), figsize=(20, 8))

psnr_values = []
mse_values = []

for i, sigma in enumerate(sigma_values):
    np.random.seed(42)
    noisy = np.clip(original + np.random.normal(0, sigma, original.shape), 0, 255)
    psnr = compute_psnr(original, noisy)
    mse = compute_mse(original, noisy)
    psnr_values.append(psnr)
    mse_values.append(mse)
    
    axes[0, i].imshow(noisy, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(f'σ = {sigma}\nPSNR = {psnr:.1f} dB', fontsize=11)
    axes[0, i].axis('off')
    
    # Difference image
    diff = np.abs(original - noisy)
    axes[1, i].imshow(diff, cmap='hot', vmin=0, vmax=100)
    axes[1, i].set_title(f'|f - g|  MSE={mse:.1f}', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Noisy Image', fontsize=12)
axes[1, 0].set_ylabel('Error Image', fontsize=12)

plt.suptitle('Effect of Noise Level on Image Quality (PSNR)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PSNR vs noise level
sigma_range = np.arange(1, 100, 1)
psnr_curve = []

for sigma in sigma_range:
    np.random.seed(42)
    noisy = np.clip(original + np.random.normal(0, sigma, original.shape), 0, 255)
    psnr_curve.append(compute_psnr(original, noisy))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sigma_range, psnr_curve, 'b-', linewidth=2)
ax.axhline(y=40, color='g', linestyle='--', alpha=0.7, label='Excellent (40 dB)')
ax.axhline(y=30, color='orange', linestyle='--', alpha=0.7, label='Good (30 dB)')
ax.axhline(y=20, color='r', linestyle='--', alpha=0.7, label='Poor (20 dB)')
ax.set_xlabel('Noise Standard Deviation (σ)', fontsize=12)
ax.set_ylabel('PSNR (dB)', fontsize=12)
ax.set_title('Peak Signal-to-Noise Ratio vs Noise Level', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Overview of Restoration Approaches

Given the degradation model $G(u,v) = H(u,v) \cdot F(u,v) + N(u,v)$, restoration methods can be categorized:

### When only noise is present ($H = \delta$, so $G = F + N$):
| Method | Domain | Key Idea |
|---|---|---|
| Mean filters | Spatial | Average neighborhood pixels |
| Order-statistic filters | Spatial | Use rank ordering (e.g., median) |
| Adaptive filters | Spatial | Adjust behavior based on local statistics |
| Band-reject/Notch filters | Frequency | Remove periodic noise frequencies |

### When blur is present ($H \neq \delta$):
| Method | Key Idea | Limitation |
|---|---|---|
| Inverse filtering | $\hat{F} = G / H$ | Noise amplification where $H \approx 0$ |
| Wiener filtering | Minimize $E\{|f - \hat{f}|^2\}$ | Needs noise-to-signal ratio |
| Constrained Least Squares | Minimize $\|\nabla^2 \hat{f}\|^2$ | Needs noise variance |
| Lucy-Richardson | Iterative ML estimation | Needs PSF, slow |

### These methods will be covered in the following lessons.

In [ ]:
# Preview: Why naive inverse filtering fails

# Degrade with Gaussian blur + noise
psf = pad_psf_to_image_size(create_gaussian_psf(64, sigma=3), (M, N))
blurred = apply_degradation(original, psf)
np.random.seed(42)
degraded = np.clip(blurred + np.random.normal(0, 5, blurred.shape), 0, 255)

# Attempt naive inverse filtering: F_hat = G / H
G = np.fft.fft2(degraded)
H = np.fft.fft2(psf)

# Direct inverse (will amplify noise)
H_safe = np.where(np.abs(H) > 1e-10, H, 1e-10)  # avoid division by zero
F_hat = G / H_safe
restored_naive = np.real(np.fft.ifft2(F_hat))
restored_naive = np.clip(restored_naive, 0, 255)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original f(x,y)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(degraded, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Degraded g(x,y)\n(blur + noise)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(restored_naive, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Naive Inverse Filter\n$\hat{F} = G / H$ (FAILS!)', fontsize=12)
axes[2].axis('off')

plt.suptitle('Why Simple Inverse Filtering Fails: Noise Amplification',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Naive inverse filtering amplifies noise where H(u,v) is small!")
print(f"PSNR of degraded: {compute_psnr(original, degraded):.1f} dB")
print(f"PSNR of naive inverse: {compute_psnr(original, restored_naive):.1f} dB")
print("The 'restored' image is WORSE than the degraded image!")
print("\nThis motivates the need for better restoration methods (Wiener, CLS, etc.)")

In [ ]:
# Explain WHY naive inverse fails: show 1/H amplification

H_centered = np.fft.fftshift(np.fft.fft2(psf))
H_magnitude = np.abs(H_centered)
H_inverse = 1.0 / np.maximum(H_magnitude, 1e-10)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(H_magnitude, cmap='hot')
axes[0].set_title('|H(u,v)| Degradation', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(np.log1p(H_inverse), cmap='hot')
axes[1].set_title('log(1 + |1/H(u,v)|) Inverse', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Cross-section
center = M // 2
axes[2].plot(H_magnitude[center, center:], 'b-', linewidth=2, label='|H(u,v)|')
axes[2].plot(np.minimum(H_inverse[center, center:], 100), 'r-', linewidth=2, label='|1/H(u,v)|')
axes[2].set_xlabel('Distance from center', fontsize=12)
axes[2].set_ylabel('Magnitude', fontsize=12)
axes[2].set_title('Cross-Section: H vs 1/H', fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.suptitle('The Problem: Where H→0, 1/H→∞ (Noise Amplification)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("At high frequencies, H(u,v) → 0 (the blur attenuates high frequencies)")
print("Inverse filter 1/H → ∞ at these same frequencies")
print("Any noise present at those frequencies gets amplified enormously!")
print("This is the fundamental challenge of image restoration.")

---
## Summary

What we learned:

1. **Image restoration** is an objective process that uses a mathematical model to recover degraded images, unlike enhancement which is subjective.

2. **The degradation model**: $g(x,y) = h(x,y) \star f(x,y) + \eta(x,y)$, equivalently $G(u,v) = H(u,v) \cdot F(u,v) + N(u,v)$ in the frequency domain.

3. **Point Spread Function (PSF)** $h(x,y)$ describes how the system blurs the image. Common types: Gaussian, motion, disk (defocus).

4. **Additive noise** $\eta(x,y)$ corrupts the image with random values. Common types: Gaussian, salt-and-pepper, Poisson.

5. **PSNR** measures image quality: higher is better. PSNR > 40 dB is excellent, < 20 dB is very poor.

6. **Naive inverse filtering** ($\hat{F} = G/H$) fails because it amplifies noise at frequencies where $H(u,v) \approx 0$.

7. **Better restoration methods** (Wiener filter, Constrained Least Squares) are needed to balance deblurring and noise suppression.